# Exploratory Data Analysis (EDA)

Exploratory Data Analysis (EDA) is performed to understand the characteristics of the engineered flight dataset and identify patterns associated with flight delays.

The objectives of this notebook are to:

- Explore the distribution of important variables.
- Identify relationships between features and the target variable.
- Discover operational trends affecting flight delays.
- Generate business insights that support the machine learning phase.

The analysis includes both univariate and bivariate visualizations together with interpretations of the observed patterns.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)

In [ ]:
from pathlib import Path


def find_project_root():
    """
    Locate the project root regardless of whether the notebook
    is launched from the project root or the notebooks folder.
    """
    current_path = Path.cwd().resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root could not be located. "
        "Keep the data and notebooks folders inside the same project folder."
    )


PROJECT_ROOT = find_project_root()

RAW_DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

PROCESSED_DATA_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

MODELS_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True
)

print("Project root:", PROJECT_ROOT)

In [ ]:
engineered_data_path = (
    PROCESSED_DATA_DIRECTORY
    / "engineered_flight_data.csv"
)

df = pd.read_csv(
    engineered_data_path,
    parse_dates=["FL_DATE"]
)

print("Loaded from:", engineered_data_path)
print("Dataset shape:", df.shape)

df.head()

In [ ]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

In [ ]:
df.info()

### 📝 Interpretation

The engineered dataset was successfully loaded and contains the features created during the preprocessing and feature engineering stages.

This dataset will be explored to identify meaningful trends and relationships that may improve predictive model performance.

# Business Question 1

## What percentage of flights experience a significant arrival delay?

Understanding the proportion of delayed flights provides an overview of the target variable and helps identify whether the dataset is balanced.

This information is important because class imbalance may influence model selection and evaluation.

In [ ]:
delay_counts = df["IS_DELAYED"].value_counts().sort_index()

delay_labels = ["On Time", "Delayed"]

plt.figure(figsize=(6,5))

plt.bar(
    delay_labels,
    delay_counts.values
)

plt.title("Distribution of Flight Delay Status")
plt.xlabel("Flight Status")
plt.ylabel("Number of Flights")

plt.show()

In [ ]:
delay_summary = pd.DataFrame({
    "Count": df["IS_DELAYED"].value_counts().sort_index(),
    "Percentage (%)": (
        df["IS_DELAYED"]
        .value_counts(normalize=True)
        .sort_index()
        *100
    ).round(2)
})

delay_summary.index = ["On Time","Delayed"]

delay_summary

### 📝 Interpretation

Most flights arrive on time or with only a minor delay.

Approximately **17%** of flights experience an arrival delay greater than 15 minutes, while the remaining flights arrive on time.

Although the dataset is moderately imbalanced, the delayed class still contains a large number of observations, making it suitable for machine learning classification.

### 💡 Business Insight

Only a minority of flights experience significant delays.

Predicting this minority class accurately is valuable because delayed flights can increase operational costs, passenger dissatisfaction, and scheduling disruptions.

# Business Question 2

## Which airlines operate the largest number of flights?

Understanding the distribution of flights across airlines provides context for later comparisons of delay performance.

In [ ]:
airline_counts = (
    df["AIRLINE"]
    .value_counts()
)

plt.figure(figsize=(12,6))

airline_counts.plot(kind="bar")

plt.title("Number of Flights by Airline")
plt.xlabel("Airline")
plt.ylabel("Number of Flights")

plt.xticks(rotation=90)

plt.show()

In [ ]:
airline_counts.head(10)

### 📝 Interpretation

Flight operations are not evenly distributed across airlines.

Some airlines operate substantially more flights than others, indicating that the dataset contains varying levels of representation across carriers.

### 💡 Business Insight

Airlines with larger flight volumes contribute more heavily to the dataset and may have a greater impact on overall delay patterns.

When comparing airline performance, delay **rates** should be considered rather than only the total number of delayed flights.

# Business Question 3

## At what time of day do most flights depart?

This analysis examines the distribution of departure hours to identify the busiest periods of the day.

Understanding departure patterns helps explain airport congestion and operational workload.

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(
    df["DEP_HOUR"],
    bins=24,
    edgecolor="black"
)

plt.title("Distribution of Departure Hours")
plt.xlabel("Departure Hour")
plt.ylabel("Number of Flights")

plt.xticks(range(0,24))

plt.show()

### 📝 Interpretation

Flight departures are concentrated during daytime hours, with noticeable peaks during the morning and afternoon.

Very few flights depart during late-night hours, reflecting normal airline scheduling practices.

### 💡 Business Insight

Peak departure periods correspond to the busiest airport operations. Increased traffic during these hours may contribute to delays because of runway congestion and higher demand for airport resources.

# Business Question 4

## How are flights distributed by travel distance?

This analysis explores the distribution of flight distances and identifies whether short-haul or long-haul flights dominate the dataset.

In [ ]:
plt.figure(figsize=(10,5))

plt.hist(
    df["DISTANCE"],
    bins=40,
    edgecolor="black"
)

plt.title("Distribution of Flight Distance")
plt.xlabel("Distance (Miles)")
plt.ylabel("Number of Flights")

plt.show()

In [ ]:
df["DISTANCE"].describe()

### 📝 Interpretation

The distribution is right-skewed, indicating that most flights cover relatively short distances while fewer flights travel very long distances.

### 💡 Business Insight

Since short-haul flights represent the majority of operations, improvements in their punctuality could have the greatest overall impact on airline performance and customer satisfaction.

# Business Question 5

## How are flights distributed across seasons?

Seasonal analysis helps determine whether flight activity varies throughout the year, which may influence operational planning and delay prediction.

In [ ]:
season_counts = (
    df["SEASON"]
    .value_counts()
    .sort_index()
)

plt.figure(figsize=(7,7))

plt.pie(
    season_counts,
    labels=season_counts.index,
    autopct="%1.1f%%",
    startangle=90
)

plt.title("Flight Distribution by Season")

plt.show()

### 📝 Interpretation

Flights are distributed across all seasons, although some seasons account for a larger share of total operations than others. These differences may reflect travel demand and airline scheduling.

### 💡 Business Insight

Seasonal travel demand may influence airport congestion and aircraft utilization, making seasonality an important feature for delay prediction.

# Business Question 6

## Which airlines experience the highest delay rates?

This analysis compares the percentage of delayed flights among the ten airlines with the highest number of flights.

Using delay rate instead of total delayed flights provides a fair comparison regardless of airline size.

In [ ]:
# Top 10 airlines by number of flights
top_airlines = df["AIRLINE"].value_counts().nlargest(10).index

delay_rate = (
    df[df["AIRLINE"].isin(top_airlines)]
    .groupby("AIRLINE")["IS_DELAYED"]
    .mean()
    .sort_values(ascending=False) * 100
)

plt.figure(figsize=(10,6))

delay_rate.plot(kind="bar")

plt.title("Delay Rate by Airline (Top 10 Airlines)")
plt.xlabel("Airline")
plt.ylabel("Delayed Flights (%)")

plt.xticks(rotation=45)

plt.show()

In [ ]:
delay_rate.round(2)

Delay rates vary noticeably across airlines.

Some airlines consistently record a higher percentage of delayed flights, suggesting operational differences such as scheduling efficiency, route networks, or airport congestion.

Delay rate is a more meaningful performance indicator than the total number of delayed flights.

Airlines with higher delay rates may benefit from operational improvements and more accurate delay prediction systems.

# Business Question 7

## Does the time of departure influence flight delays?

Departure time may affect flight punctuality because congestion accumulates throughout the day.

This analysis compares delay rates across different periods of the day.

In [ ]:
time_delay = (
    df.groupby("TIME_OF_DAY")["IS_DELAYED"]
    .mean()
    *100
).sort_values()

plt.figure(figsize=(8,5))

time_delay.plot(kind="bar")

plt.title("Delay Rate by Time of Day")
plt.xlabel("Time of Day")
plt.ylabel("Delayed Flights (%)")

plt.xticks(rotation=0)

plt.show()

In [ ]:
time_delay.round(2)

Delay rates differ across the day.

Flights departing later in the day often experience higher delays because delays from earlier flights can propagate through aircraft rotations and airport operations.

Departure time appears to be an important predictor of delays and should be retained as an input feature for machine learning models.

# Business Question 8

## Does flight distance influence delay frequency?

This analysis investigates whether short-haul and long-haul flights experience different delay rates.

In [ ]:
distance_delay = (
    df.groupby("DISTANCE_CATEGORY")["IS_DELAYED"]
    .mean()
    *100
)

plt.figure(figsize=(8,5))

distance_delay.plot(kind="bar")

plt.title("Delay Rate by Distance Category")
plt.xlabel("Distance Category")
plt.ylabel("Delayed Flights (%)")

plt.xticks(rotation=0)

plt.show()

In [ ]:
distance_delay.round(2)

Delay rates vary across distance categories, indicating that flight length may influence operational performance.

Flight distance can provide useful predictive information because operational challenges differ between short-haul and long-haul routes.

# Business Question 9

## Does seasonality influence flight delays?

Travel demand changes throughout the year because of holidays, weather conditions, and tourism.

This analysis examines whether delay rates vary by season.

In [ ]:
season_delay = (
    df.groupby("SEASON")["IS_DELAYED"]
    .mean()
    *100
)

plt.figure(figsize=(8,5))

season_delay.plot(kind="bar")

plt.title("Delay Rate by Season")
plt.xlabel("Season")
plt.ylabel("Delayed Flights (%)")

plt.xticks(rotation=0)

plt.show()

In [ ]:
season_delay.round(2)

# 📝 Interpretation
Seasonal differences in delay rates suggest that weather conditions and travel demand influence flight punctuality.

# 💡 Business Insight
Seasonality should be considered in predictive models because certain seasons are associated with increased operational complexity.

# Business Question 10

## Which numerical variables are most strongly correlated?

A correlation heatmap helps identify relationships among numerical variables and highlights features that may be useful for predictive modeling.

In [ ]:
numeric_df = df.select_dtypes(include=["number"])

plt.figure(figsize=(14,10))

sns.heatmap(
    numeric_df.corr(),
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap")

plt.show()

# 📝 Interpretation
Most numerical variables exhibit weak to moderate correlations.

This indicates that the dataset contains relatively independent predictors, which is generally beneficial for machine learning because it reduces redundancy among features.

# 💡 Business Insight
The correlation analysis helps identify useful predictors while also highlighting variables that may provide overlapping information. These findings will guide feature selection and model development.

In [ ]:
delay_by_day = (
    df.groupby("DAY_OF_WEEK")["IS_DELAYED"]
      .mean() * 100
)

plt.figure(figsize=(8,5))
delay_by_day.plot(kind="bar")
plt.title("Delay Rate by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Delayed Flights (%)")
plt.xticks(
    range(7),
    ["Mon","Tue","Wed","Thu","Fri","Sat","Sun"],
    rotation=0
)
plt.show()

In [ ]:
monthly_delay = (
    df.groupby("MONTH")["IS_DELAYED"]
      .mean() * 100
)

plt.figure(figsize=(10,5))
monthly_delay.plot(marker="o")
plt.title("Monthly Delay Rate")
plt.xlabel("Month")
plt.ylabel("Delayed Flights (%)")
plt.grid(True)
plt.show()

In [ ]:
weekend_delay = (
    df.groupby("IS_WEEKEND")["IS_DELAYED"]
      .mean() * 100
)

plt.figure(figsize=(6,5))
weekend_delay.plot(kind="bar")
plt.xticks([0,1],["Weekday","Weekend"],rotation=0)
plt.ylabel("Delayed Flights (%)")
plt.title("Weekend vs Weekday Delay Rate")
plt.show()

In [ ]:
peak_delay = (
    df.groupby("IS_PEAK_SEASON")["IS_DELAYED"]
      .mean() * 100
)

plt.figure(figsize=(6,5))
peak_delay.plot(kind="bar")
plt.xticks([0,1],["Non-Peak","Peak"],rotation=0)
plt.ylabel("Delayed Flights (%)")
plt.title("Delay Rate During Peak Seasons")
plt.show()

In [ ]:
busy_delay = (
    df.groupby("IS_BUSY_HOUR")["IS_DELAYED"]
      .mean() * 100
)

plt.figure(figsize=(6,5))
busy_delay.plot(kind="bar")
plt.xticks([0,1],["Normal Hours","Busy Hours"],rotation=0)
plt.ylabel("Delayed Flights (%)")
plt.title("Delay Rate During Busy Hours")
plt.show()

In [ ]:
selected_features = [
    "DEP_DELAY",
    "DISTANCE",
    "CRS_ELAPSED_TIME",
    "DEP_HOUR",
    "MONTH",
    "QUARTER",
    "IS_WEEKEND",
    "IS_PEAK_SEASON",
    "IS_BUSY_HOUR",
    "IS_DELAYED"
]

corr = df[selected_features].corr()

plt.figure(figsize=(10, 8))

sns.heatmap(
    corr,
    annot=True,
    cmap="coolwarm",
    fmt=".2f",
    square=True
)

plt.title("Correlation Matrix of Selected Features")

plt.show()

# Key Findings

- Approximately 17% of flights are delayed by more than 15 minutes.
- Flight operations are concentrated during daytime hours.
- Delay rates vary across airlines.
- Departure time influences the likelihood of delays.
- Flight distance and seasonality appear to affect delay frequency.
- Numerical features generally exhibit low multicollinearity, making them suitable for predictive modeling.

# Notebook Summary

This notebook explored the engineered flight dataset using both univariate and bivariate analyses.

Several operational factors—including airline, departure time, seasonality, travel distance, and airport activity—were found to influence flight delays.

The insights obtained from this analysis provide a strong foundation for selecting the most informative features and developing machine learning models capable of predicting flight delays accurately.

# Overall EDA Summary

The exploratory analysis revealed several important patterns in the flight dataset.

### Main Findings

- Approximately 17% of flights arrive more than 15 minutes late.
- Flight departures are concentrated during daytime operating hours.
- Delay rates vary considerably among airlines.
- Flights departing later in the day generally experience higher delay rates.
- Seasonal and monthly variations suggest that weather and travel demand influence flight punctuality.
- Busy operating periods show a higher likelihood of delays.
- Most numerical predictors exhibit relatively low correlation with one another, indicating limited multicollinearity.

### Implications for Machine Learning

The exploratory analysis confirms that several engineered variables—including departure hour, season, peak season, busy hour, and distance—are likely to contribute useful predictive information.

These insights provide a strong justification for proceeding with feature selection and predictive model development.